In [2]:
import requests
import pandas as pd
from pathlib import Path
import time

def load_api_key(filepath="api_keys/twelvedata.txt"):
    return Path(filepath).read_text(encoding="utf-8").strip()


def get_twelve_data_hourly(ticker, api_key, outputsize=5000):
    url = "https://api.twelvedata.com/time_series"

    params = {
        "symbol": ticker,
        "interval": "1h",
        "outputsize": outputsize,
        "apikey": api_key,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    if "values" not in data:
        raise ValueError(f"Twelve Data error for {ticker}: {data}")

    df = pd.DataFrame(data["values"])

    df = df.rename(columns={
        "datetime": "timestamp",
        "open": "open",
        "high": "high",
        "low": "low",
        "close": "close",
        "volume": "volume",
    })

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["ticker"] = ticker

    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["vwap"] = pd.NA
    df["transactions"] = pd.NA

    df = df[
        ["timestamp", "ticker", "open", "high", "low", "close", "volume", "vwap", "transactions"]
    ]

    return df.sort_values("timestamp").reset_index(drop=True)


def download_twelve_data_hourly_for_tickers(
    tickers,
    api_key_path="api_keys/twelve_data.txt",
    save_path="data/raw_hourly_data.csv",
    sleep_seconds=8
):
    api_key = load_api_key(api_key_path)
    all_dfs = []

    for ticker in tickers:
        print(f"Downloading {ticker}...")
        df = get_twelve_data_hourly(ticker, api_key)
        all_dfs.append(df)
        time.sleep(sleep_seconds)

    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df = combined_df.sort_values(["ticker", "timestamp"]).reset_index(drop=True)

    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    combined_df.to_csv(save_path, index=False)

    print(f"Saved {len(combined_df):,} rows to {save_path}")

    return combined_df

In [ ]:
tickers = [
    # Big Tech / Growth
    "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "TSLA", "NFLX", "META",
    
    # Finance
    "JPM", "BAC", "GS", "MS",
    
    # Consumer / Retail
    "WMT", "COST", "HD", "NKE", "SBUX",
    
    # Healthcare / Pharma
    "JNJ", "PFE", "MRK", "UNH",
    
    # Energy
    "XOM", "CVX",
    
    # Industrials / Transportation
    "BA", "CAT", "GE", "UPS",
    
    # ETFs (nice for comparison)
    "SPY", "QQQ", "DIA"
]
hourly_df = download_twelve_data_hourly_for_tickers(
    tickers=tickers,
    api_key_path="api_keys/twelvedata.txt",
    save_path="data/raw_hourly_data.csv",
    sleep_seconds=8
)

hourly_df.head()

In [6]:
hourly_df['ticker'].value_counts()

ticker
AAPL     5000
AMZN     5000
GOOGL    5000
MSFT     5000
NVDA     5000
Name: count, dtype: int64